In [1]:
%load_ext autoreload
%autoreload 2

from uuid import uuid4
from utils.aimodel import load_llm_conf, load_llm_lc
from utils.file import wr, wrp, rdp
from utils.json_polyfill import install_json
from db.store.cacheservice import *
install_json()

In [2]:
dataset = Path("./dataset")
dataset.mkdir(exist_ok=True)
ds_control = dataset / "control"
ds_control.mkdir(exist_ok=True)
ds_files = dataset / "file"
ds_files.mkdir(exist_ok=True)
ds_res = dataset / "output"
ds_res.mkdir(exist_ok=True)
ds_pdf = dataset / "pdf"
ds_pdf.mkdir(exist_ok=True)

In [3]:
# 5e, a7
from validate.generator.kg import make_new_kg

for i in range(1):
    uid = str(uuid4())
    kg, nxg = make_new_kg(10, 10, 10)
    # kg, nxg = make_new_kg(70, 150, 100)
    wrp(ds_control / uid, dict(kg=kg, nxg=nxg))

In [5]:
from validate.generator.doc_plan import plan_corpus

for i in ds_control.iterdir():
    data = rdp(i)
    uid = i.name
    if data.get('plan') is None:
        data['plan'] = plan_corpus(data['kg'])
        wrp(i, data)

In [10]:
from validate.generator.doc_plan import generate_corpus

for i in ds_control.iterdir():
    data = rdp(i)
    if data.get('docs') is None:
        out = ds_files / i.name
        out.mkdir(exist_ok=True)
        data['docs'] = await generate_corpus(data['kg'], data['plan'], out)
        wrp(i, data)

In [11]:
from playwright.async_api import async_playwright

async def render_pdf(html_path: Path, pdf_path: Path) -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(html_path.resolve().as_uri())
        await page.pdf(path=str(pdf_path))
        await browser.close()


In [12]:
from docling.datamodel.base_models import InputFormat
from document.docling.provider import docling_provider

dumb_docling = docling_provider(use_vlm=False, use_ocr=False)

for i in ds_control.iterdir():
    out = ds_files / i.name
    out.mkdir(exist_ok=True)
    f_pdf_dir = ds_pdf / i.name
    f_pdf_dir.mkdir(exist_ok=True)
    f_out_dir = ds_res / i.name
    f_out_dir.mkdir(exist_ok=True)
    for f_in in out.iterdir():
        if not f_in.name.endswith(".html"): continue

        f_out = f_pdf_dir / f_in.name.replace(".html", ".pdf")
        if not f_out.exists():
            await render_pdf(f_in, f_out)

        f_out = f_out_dir / f_in.name.replace(".html", ".md")
        if not f_out.exists():
            doc = dumb_docling.convert_string(f_in.read_text("utf-8"), format=InputFormat.HTML).document
            wr(f_out, doc.export_to_markdown())


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


In [13]:
from validate.generator.question import make_questions

for i in ds_control.iterdir():
    data = rdp(i)
    if data.get('questions') is None:
        res = await make_questions(data['kg'], data['plan'])
        data['questions'] = res
        wrp(i, data)

In [7]:
from validate.generator.doc_plan import CorpusPlan, GeneratedDocument
from validate.generator.kg import KGGen
from validate.generator.question import GeneratedQuestion
from validate.generator.ragdataset import *

dataset = KGRAGDataset()

ds_control = Path("dataset/control")
for batch in ds_control.iterdir():
    data = rdp(batch)
    kg: KGGen = data['kg']
    qs: List[GeneratedQuestion] = data['questions']
    plan: CorpusPlan = data['plan']
    docs: List[GeneratedDocument] = data['docs']

    blk = KGRAGDatasetBlock(
        block_id=batch.name,
        block_context=kg.domain_overview,
        document_names=[f"dataset/pdf/{batch.name}/{i.filename.replace('html', 'pdf')}"
                        for i in plan.documents],
        document_contexts=[i.html for i in docs]
    )

    for q in qs:
        blk.questions.append(KGRAGDatasetQuestion(
            type=q.type,
            is_unanswerable=q.type == "irresolvable",
            is_trick=q.type in {"irresolvable", "inconsistency"},
            question=q.question,
            answer=q.answer,
            golden_claims=[
                              KGRAGClaim(type="claim", mode=i.mode, claim=i.text)
                              for i in q.atoms
                          ] + [
                              KGRAGClaim(type="entity", mode="extracted", claim=e.canonical_name)
                              for i in q.golden_entity_uids if (e := kg.entities.get(i))
                          ]
        ))

    dataset.document_blocks.append(blk)

wr("dataset/data.json", dataset.model_dump_json(ensure_ascii=False))

In [17]:
len(dataset.document_blocks[1].questions)

110